# Credit Default — Final Colab Reproducible


REPRODUCTION_ONLY. This notebook reproduces the frozen A1 candidate.


No tuning, model selection, threshold search, or feature experimentation is performed.

## 1. Objective and A1 requirements


The target is `default = 1`. The frozen candidate is CatBoostClassifier with A3 = ROUND1 + BILL + PAYMENT and threshold 0.247743.


Official A1 gates: ROC-AUC >= 0.75, Recall(default=1) >= 0.60, and Macro F1 >= 0.65. Binary F1(default=1) remains a diagnostic.

## 2. Environment setup


Run this section once in a fresh CPU Colab runtime. The repository URL is the only publication-time field that may need filling.

In [ ]:
import os

import subprocess

from pathlib import Path


REPO_URL = "https://github.com/EngIaCeub/Machine_Learning_Inadimplencia.git"

REPO_DIR = Path("/content/Machine_Learning_Inadimplencia")

os.chdir("/content")


if not REPO_DIR.exists():

    if False:

        raise RuntimeError("Fill REPO_URL once with the public read-only repository URL.")

    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)


os.chdir(REPO_DIR)

%pip install -q -r requirements.txt catboost==1.2.10
%pip install -q -e .

os.environ["PYTHONPATH"] = str(Path.cwd() / "src")

print("Repository:", REPO_URL)
print("Working directory:", Path.cwd())
print("Environment setup complete.")

## 3. Imports and reproducibility

All randomness is centralized at seed 42.

In [ ]:
import json

import random

import sys

from pathlib import Path


import catboost

import matplotlib.pyplot as plt

import numpy as np

import pandas as pd

import sklearn

from catboost import CatBoostClassifier

from sklearn.metrics import (

    ConfusionMatrixDisplay, average_precision_score, confusion_matrix,

    precision_recall_curve, roc_auc_score, roc_curve,

)


SEED = 42

random.seed(SEED)

np.random.seed(SEED)

print("Python:", sys.version)

print("pandas:", pd.__version__)

print("numpy:", np.__version__)

print("scikit-learn:", sklearn.__version__)

print("catboost:", catboost.__version__)

print("Random seed:", SEED)

## 4. Dataset loading

The official UCI loader downloads dataset id 350 programmatically. No manual upload or personal storage is used.

In [ ]:
sys.path.insert(0, str(Path.cwd() / "src"))

from credit_default.config import get_project_config

from credit_default.data.load import load_uci_dataset

from credit_default.data.schema import validate_basic_shape

from credit_default.data.split import validate_distinct_splits

from credit_default.features.credit_default_features import (

    RAW_CATEGORICAL_COLUMNS, build_behavioral_features,

)

from credit_default.modeling.evaluate import evaluate_binary_classifier


config = get_project_config()

features, target = load_uci_dataset(dataset_id=350)

validate_basic_shape(n_rows=len(features), n_features=features.shape[1])

print("Dataset shape:", features.shape)

display(features.head())

display(target.value_counts().sort_index().rename("count").to_frame())

display(features.isna().sum().sort_values(ascending=False).head(10).rename("missing").to_frame())

## 5. Dataset overview and target definition

There are 23 original attributes. The target is normalized by the official loader and must contain only 0 and 1.

In [ ]:
assert len(features) >= 29000 and len(features) <= 31000

assert set(target.unique()) == {0, 1}

assert len(features) == len(target)

assert not any(column in features.columns for column in ("Y", "default", "target"))

print("Positive class: default = 1")

print("Original features:", features.shape[1])

## 6. Frozen Train/Validation/Test split

The persisted Round 4 manifest is used directly. Test is the complement of the frozen Train and Validation indices; no new split is created.

In [ ]:
manifest_path = Path("artifacts/experiments/round4_split_manifest.json")

manifest = json.loads(manifest_path.read_text())

train_index = pd.Index(manifest["train_indices"], dtype=features.index.dtype)

validation_index = pd.Index(manifest["validation_indices"], dtype=features.index.dtype)

test_index = features.index.difference(train_index.union(validation_index))


assert train_index.intersection(validation_index).empty

assert train_index.intersection(test_index).empty

assert validation_index.intersection(test_index).empty

X_train, y_train = features.loc[train_index], target.loc[train_index]

X_validation, y_validation = features.loc[validation_index], target.loc[validation_index]

X_test, y_test = features.loc[test_index], target.loc[test_index]

split_table = pd.DataFrame({

    "split": ["train", "validation", "test"],

    "rows": [len(X_train), len(X_validation), len(X_test)],

    "positive_rate": [y_train.mean(), y_validation.mean(), y_test.mean()],

})

display(split_table)

assert split_table["rows"].tolist() == [21000, 4500, 4500]

## 7. Feature engineering A3

A3 is exactly ROUND1 + BILL + PAYMENT. PAY remains ordinal/temporal; X2, X3, and X4 are categorical for CatBoost. No target is passed to feature engineering.

In [ ]:
A3_GROUPS = ("round1", "bill", "payment")

X_train_a3 = build_behavioral_features(X_train, enabled_groups=A3_GROUPS)

X_validation_a3 = build_behavioral_features(X_validation, enabled_groups=A3_GROUPS)

X_test_a3 = build_behavioral_features(X_test, enabled_groups=A3_GROUPS)

for frame in (X_train_a3, X_validation_a3, X_test_a3):

    assert not any(column in frame.columns for column in ("Y", "default", "target"))

    assert np.isfinite(frame.select_dtypes(include="number").to_numpy()).all()

print("Original features:", features.shape[1])

print("Final A3 features:", X_train_a3.shape[1])

print("Groups: ROUND1 + BILL + PAYMENT")

## 8. Final model configuration

Configuration is loaded from the frozen winner artifact. No tuning or threshold search is performed.

In [ ]:
winner = json.loads(Path("artifacts/final/development_winner.json").read_text())

params = winner["hyperparameters"].copy()

FROZEN_THRESHOLD = float(winner["metrics"]["threshold"])

cat_columns = [column for column in RAW_CATEGORICAL_COLUMNS if column in X_train_a3.columns]

for frame in (X_train_a3, X_validation_a3, X_test_a3):

    for column in cat_columns:

        frame[column] = frame[column].astype(str)

model_summary = {

    "model": winner["model"],

    "feature_set": winner["feature_set"],

    "hyperparameters": params,

    "seed": SEED,

    "categorical_features": cat_columns,

    "frozen_threshold": FROZEN_THRESHOLD,

}

display(pd.Series(model_summary, dtype=object).to_frame("value"))

## 9. Training

The frozen CatBoost candidate is fitted only on TRAIN. The model uses native categorical handling and CPU-safe settings.

In [ ]:
model = CatBoostClassifier(

    loss_function="Logloss",

    eval_metric="AUC",

    random_seed=SEED,

    verbose=False,

    thread_count=2,

    allow_writing_files=False,

    **params,

)

model.fit(X_train_a3, y_train, cat_features=cat_columns)

print("Training complete.")

## 10. Evaluation function

The same explicit positive-class metric semantics are used for Validation and Test. Scores are probabilities; predictions use only the frozen threshold.

In [ ]:
def evaluate_split(name, X_frame, y_frame):

    scores = model.predict_proba(X_frame)[:, 1]

    metrics = evaluate_binary_classifier(y_frame, scores, FROZEN_THRESHOLD)

    metrics["average_precision"] = average_precision_score(y_frame, scores)

    metrics["split"] = name

    return scores, metrics


validation_scores, validation_metrics = evaluate_split("validation", X_validation_a3, y_validation)

test_scores, test_metrics = evaluate_split("test", X_test_a3, y_test)

metric_names = ["roc_auc", "average_precision", "precision_positive", "recall_positive", "f1_binary_default_1", "f1_class_0", "f1_macro", "f1_weighted", "accuracy", "tn", "fp", "fn", "tp"]

display(pd.DataFrame([{"metric": key, "validation": validation_metrics[key], "test": test_metrics[key]} for key in metric_names]))

## 11. Validation evaluation

Validation is shown before the final holdout to confirm the frozen candidate remains consistent with its reference evidence.

In [ ]:
validation_reference = {"roc_auc": 0.7811007964, "recall_positive": 0.6050251256, "f1_macro": 0.7024923058}

display(pd.DataFrame([{"metric": key, "reproduced": validation_metrics[key], "reference": value, "absolute_difference": abs(validation_metrics[key] - value)} for key, value in validation_reference.items()]))

## 12. Final TEST evaluation

TEST is evaluated once in this reproduction-only notebook with the frozen model and threshold. It is not used for any decision.

In [ ]:
test_display = {key: test_metrics[key] for key in metric_names}

display(pd.Series(test_display, name="test").to_frame())

print("Frozen threshold =", FROZEN_THRESHOLD)

## 13. A1 gate analysis

The official F1 gate is Macro F1. Binary F1(default=1) is retained as a diagnostic and is not hidden.

In [ ]:
gate_table = pd.DataFrame([

    {"metric": "ROC-AUC", "value": test_metrics["roc_auc"], "requirement": ">= 0.75", "status": test_metrics["roc_auc"] >= 0.75},

    {"metric": "Recall(default=1)", "value": test_metrics["recall_positive"], "requirement": ">= 0.60", "status": test_metrics["recall_positive"] >= 0.60},

    {"metric": "Macro F1", "value": test_metrics["f1_macro"], "requirement": ">= 0.65", "status": test_metrics["f1_macro"] >= 0.65},

])

gate_table["status"] = gate_table["status"].map({True: "PASS", False: "FAIL"})

display(gate_table)

print("Overall:", (gate_table["status"] == "PASS").sum(), "/ 3 PASS")

assert (gate_table["status"] == "PASS").all()

## 14. Confusion matrix

The matrix is generated from the reproduced TEST predictions at the frozen threshold.

In [ ]:
test_predictions = (test_scores >= FROZEN_THRESHOLD).astype(int)

cm = confusion_matrix(y_test, test_predictions, labels=[0, 1])

ConfusionMatrixDisplay(cm, display_labels=[0, 1]).plot(cmap="Blues", values_format="d")

plt.title("TEST Confusion Matrix")

plt.show()

print("TN, FP, FN, TP =", tuple(cm.ravel()))

## 15. ROC curve

ROC-AUC uses probability scores and does not select a threshold.

In [ ]:
fpr, tpr, _ = roc_curve(y_test, test_scores)

plt.plot(fpr, tpr, label=f"AUC = {test_metrics['roc_auc']:.4f}")

plt.plot([0, 1], [0, 1], "--", color="gray")

plt.xlabel("False positive rate")

plt.ylabel("True positive rate")

plt.title("TEST ROC Curve")

plt.legend()

plt.show()

## 16. Precision-Recall curve

The PR curve is diagnostic. The frozen threshold is marked; no new threshold is searched.

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_test, test_scores)

plt.plot(recall, precision, label=f"AP = {test_metrics['average_precision']:.4f}")

frozen_point = test_metrics["recall_positive"], test_metrics["precision_positive"]

plt.scatter(*frozen_point, color="red", label=f"frozen threshold = {FROZEN_THRESHOLD:.6f}")

plt.xlabel("Recall")

plt.ylabel("Precision")

plt.title("TEST Precision-Recall Curve")

plt.legend()

plt.show()

## 17. Model explainability

CatBoost global feature importance is computed from the fitted frozen model. Importance indicates association with model decisions, not causality.

In [ ]:
importance = pd.DataFrame({

    "feature": X_train_a3.columns,

    "importance": model.get_feature_importance(),

}).sort_values("importance", ascending=False).head(15)

display(importance)

importance.sort_values("importance").plot.barh(x="feature", y="importance", legend=False, figsize=(8, 6))

plt.title("Top 15 CatBoost Feature Importance")

plt.show()

print("Groups represented: PAY history, BILL behavior, PAYMENT behavior, and A3 aggregates.")

## 18. Monitoring strategy


Monitor schema, missing values, invalid categories, and expected ranges as data quality checks. Monitor feature distributions with PSI, KS, or an equivalent agreed measure; monitor the distribution of p(default=1) and predicted positive rate for prediction drift. When labels arrive, monitor ROC-AUC, Recall(default=1), Macro F1, Precision, and the confusion matrix. Persistent values below AUC 0.75, Recall 0.60, or Macro F1 0.65 should trigger investigation and revalidation; an isolated fluctuation does not automatically require retraining.

## 19. Pipeline visual

Dataset → Schema validation → Frozen split → Feature engineering A3 → CatBoostClassifier → predict_proba → Frozen threshold → A1 metrics → Final evaluation

## 20. Experimental history summary

The complete development rounds are persisted in the project artifacts. The table below uses the persisted A1 reranking and makes complete gate eligibility visible.

In [ ]:
history = pd.read_csv("artifacts/final/a1_macro_f1_reranking.csv")

history_column_map = {
    "round": "round", "experiment": "experiment", "model": "model",
    "feature_set": "feature_set", "selection_source": "selection_source", "roc_auc": "AUC",
    "precision_positive": "precision_default_1", "recall_positive": "recall_default_1",
    "f1_binary_default_1": "f1_binary_default_1",
    "f1_macro": "f1_macro", "f1_weighted": "f1_weighted",
    "passes_auc": "passes_auc", "passes_recall": "passes_recall",
    "passes_macro_f1": "passes_macro_f1", "passes_all_A1_gates": "passes_all_A1_gates",
    "rank_if_valid": "rank_if_valid",
}
history_view = history.rename(columns={source: target for target, source in history_column_map.items()})
history_view["passes_all_A1_gates"] = history_view["passes_all_A1_gates"].astype(bool)
history_view["eligible_for_rank"] = history_view["selection_source"].eq("VALIDATION") & history_view["passes_all_A1_gates"]
history_view["A1_status"] = np.where(history_view["passes_all_A1_gates"], "PASS", "FAIL")
valid_candidates = history_view.loc[history_view["eligible_for_rank"]].sort_values(
    ["f1_macro", "roc_auc", "recall_positive", "precision_positive"], ascending=[False, False, False, False]
)
print("Main valid candidates under the A1 methodology")
display(valid_candidates)
print("Selected development experiments")
display(history_view.sort_values(["eligible_for_rank", "passes_all_A1_gates", "f1_macro"], ascending=[False, False, False]))

xgb_a3_none = history_view.loc[
    (history_view["model"] == "XGBoostClassifier") & (history_view["feature_set"] == "A3+none")
]
print("XGBoost A3+none gate status from persisted reranking")
display(xgb_a3_none[["model", "feature_set", "selection_source", "roc_auc", "recall_positive", "f1_macro", "passes_all_A1_gates", "eligible_for_rank", "A1_status"]])
print("CatBoost A3 persisted rank:", history_view.loc[history_view["model"].eq("CatBoostClassifier") & history_view["feature_set"].eq("A3"), "rank_if_valid"].iloc[0])
print("Rank eligibility is restricted to selection_source == VALIDATION; TRAIN_OOF rows remain diagnostic only.")

## 21. Reproducibility check

Reference values are used only for a diagnostic comparison. Reproduced metrics are calculated above from the dataset, frozen split, frozen features, frozen model, and frozen threshold.

In [ ]:
reference = {"roc_auc": 0.7865276746, "recall_positive": 0.6104417671, "f1_macro": 0.7019172459, "f1_binary_default_1": 0.5502262443}

repro_table = pd.DataFrame([{"metric": key, "reproduced": test_metrics[key], "reference": value, "absolute_difference": abs(test_metrics[key] - value), "status": "PASS" if abs(test_metrics[key] - value) <= 0.01 else "REVIEW"} for key, value in reference.items()])

display(repro_table)

## 22. Final conclusion


The frozen CatBoostClassifier using A3 features (ROUND1 + BILL + PAYMENT) reproduced the expected final holdout performance. On TEST, the reproduced evaluation satisfies the three A1 requirements: ROC-AUC >= 0.75, Recall(default=1) >= 0.60, and Macro F1 >= 0.65. Overall result: 3/3 A1 gates PASS. Binary F1(default=1) remains a class-specific diagnostic metric and is not used as the official A1 F1 gate. The model, A3 feature set, threshold and metric definition were frozen before final evaluation. No tuning, model selection, feature selection or threshold optimization was performed using TEST. The holdout is not written to official artifacts by this notebook. Rank eligibility is restricted to candidates with selection_source == VALIDATION; TRAIN_OOF candidates such as XGBoost A3+none remain diagnostic and do not compete for the development rank.

In [ ]:
final_summary = pd.DataFrame([
    {"metric": "ROC-AUC", "test": test_metrics["roc_auc"], "requirement": ">= 0.75", "status": "PASS" if test_metrics["roc_auc"] >= 0.75 else "FAIL"},
    {"metric": "Recall(default=1)", "test": test_metrics["recall_positive"], "requirement": ">= 0.60", "status": "PASS" if test_metrics["recall_positive"] >= 0.60 else "FAIL"},
    {"metric": "Macro F1", "test": test_metrics["f1_macro"], "requirement": ">= 0.65", "status": "PASS" if test_metrics["f1_macro"] >= 0.65 else "FAIL"},
    {"metric": "Binary F1(default=1)", "test": test_metrics["f1_binary_default_1"], "requirement": "Diagnostic", "status": "-"},
])
display(final_summary)
print("FINAL STATUS: 3/3 A1 GATES PASS" if all(final_summary.loc[:2, "status"] == "PASS") else "FINAL STATUS: REVIEW")